# Score Random 2000 Alignments with StructuralSimilarityModel

This notebook loads the saved random-pair alignments, computes structural/semantic similarity features
using `StructuralSimilarityModel`, and saves the scored labels for downstream embedding-model training.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'src').exists() and (c / 'data').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.'
    )

project_root = find_project_root()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from structural_similarity_model import StructuralSimilarityModel

alignment_path = project_root / 'data' / 'Alignment' / 'asq_random_2000_pair_alignments.json'
out_json = project_root / 'data' / 'Alignment' / 'asq_random_2000_pair_structural_scores.json'
out_csv = project_root / 'data' / 'Alignment' / 'asq_random_2000_pair_structural_scores.csv'

print('project_root:', project_root)
print('alignment_path exists:', alignment_path.exists(), '|', alignment_path)
print('output json:', out_json)
print('output csv :', out_csv)


project_root: /Users/shayan/Projects/NarrativeSimilarity
alignment_path exists: True | /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_2000_pair_alignments.json
output json: /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_2000_pair_structural_scores.json
output csv : /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_2000_pair_structural_scores.csv


In [3]:
with open(alignment_path, 'r', encoding='utf-8') as f:
    alignments = json.load(f)

print('Loaded alignment records:', len(alignments))
if len(alignments) > 0:
    print('Sample keys:', list(alignments[0].keys()))

Loaded alignment records: 2000
Sample keys: ['pair_id', 'story_a', 'story_b', 'model_output_raw', 'alignment', 'ok', 'error', 'model', 'temperature']


In [5]:
hf_cache_model_dir = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--sentence-transformers--all-MiniLM-L6-v2'
if hf_cache_model_dir.exists():
    snapshot_dirs = sorted((hf_cache_model_dir / 'snapshots').glob('*'))
    if len(snapshot_dirs) == 0:
        raise RuntimeError(f"No local snapshots found in: {hf_cache_model_dir / 'snapshots'}")
    embedding_model_path = str(snapshot_dirs[-1])
else:
    embedding_model_path = 'sentence-transformers/all-MiniLM-L6-v2'

print('Embedding model path:', embedding_model_path)
model = StructuralSimilarityModel(embedding_model_name=embedding_model_path)

scored_rows = []
for item in tqdm(alignments, desc='Scoring alignments'):
    story_a = item.get('story_a') or {}
    story_b = item.get('story_b') or {}

    row = {
        'alignment': item.get('alignment') or {},
        'EventsA_align': story_a.get('events') or [],
        'EventsB_align': story_b.get('events') or [],
    }

    pred = model.predict_similarity(row)

    scored_rows.append({
        'pair_id': item.get('pair_id'),
        'story_a_id': story_a.get('id'),
        'story_b_id': story_b.get('id'),
        'ok': item.get('ok'),
        'error': item.get('error'),
        'num_events_a': len(row['EventsA_align']),
        'num_events_b': len(row['EventsB_align']),
        'num_matches': len((row['alignment'] or {}).get('matches', []) or []),
        'D_alignment': pred['D_alignment'],
        'alignment_similarity': pred['alignment_similarity'],
        'D_semantic': pred['D_semantic'],
        'semantic_similarity': pred['semantic_similarity'],
        'pred_event_rating_mean_joint': pred['pred_event_rating_mean_joint'],
    })

scores_df = pd.DataFrame(scored_rows)
print('Scored rows:', len(scores_df))
scores_df.head()

Embedding model path: /Users/shayan/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /Users/shayan/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring alignments:   0%|          | 0/2000 [00:00<?, ?it/s]

Scored rows: 2000


,pair_id,story_a_id,story_b_id,ok,error,num_events_a,num_events_b,num_matches,D_alignment,alignment_similarity,D_semantic,semantic_similarity,pred_event_rating_mean_joint
0,9rprs0__3986p3,9rprs0,3986p3,True,NaN,2,1,0,1.0000,0.0000,1.000000,0.000000,1.583000
1,9n9qs4__9dtapa,9n9qs4,9dtapa,True,NaN,3,7,3,0.1875,0.8125,0.450051,0.549949,2.260804
2,5w3bgs__7au7vs,5w3bgs,7au7vs,True,NaN,1,6,0,1.0000,0.0000,1.000000,0.000000,1.583000
3,30ynhu__3l0qyk,30ynhu,3l0qyk,True,NaN,2,1,0,1.0000,0.0000,1.000000,0.000000,1.583000
4,6uf7ty__9jvbv3,6uf7ty,9jvbv3,True,NaN,8,3,0,1.0000,0.0000,1.000000,0.000000,1.583000


In [6]:
print('Summary statistics:')
print(scores_df[['D_alignment', 'alignment_similarity', 'D_semantic', 'semantic_similarity', 'pred_event_rating_mean_joint']].describe().to_string())

print('Failed source alignments (if any):', int((scores_df['ok'] == False).sum()))

Summary statistics:
       D_alignment  alignment_similarity   D_semantic  semantic_similarity  pred_event_rating_mean_joint
count  2000.000000           2000.000000  2000.000000          2000.000000                   2000.000000
mean      0.891825              0.108175     0.862968             0.137032                      1.732837
std       0.233952              0.233952     0.261054             0.261054                      0.285406
min       0.000000              0.000000     0.132758             0.000000                      1.583000
25%       1.000000              0.000000     1.000000             0.000000                      1.583000
50%       1.000000              0.000000     1.000000             0.000000                      1.583000
75%       1.000000              0.000000     1.000000             0.000000                      1.583000
max       1.000000              1.000000     1.000000             0.867242                      2.469940
Failed source alignments (if any): 

In [7]:
out_json.parent.mkdir(parents=True, exist_ok=True)

scores_df.to_json(out_json, orient='records', indent=2, force_ascii=False)
scores_df.to_csv(out_csv, index=False)

print('Saved JSON:', out_json)
print('Saved CSV :', out_csv)

Saved JSON: /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_2000_pair_structural_scores.json
Saved CSV : /Users/shayan/Projects/NarrativeSimilarity/data/Alignment/asq_random_2000_pair_structural_scores.csv
